# Research HB product fields

In [ ]:
import notebook_header
from notebook_header import start_supplier, \
pprint, jprint, logger, j_dumps, j_loads, \
Driver, execute_locator, Supplier, Product, ProductFields, \
SN, SF, Path, gs
import re, shutil
from selenium.webdriver.remote.webelement import WebElement

## Старт hbdeadsea.co.il

In [ ]:
s = start_supplier('hb', 'en')
""" s - на протяжении всего кода означает запущенный инстанс класса `Supplier` """


## Соглашения: 
-  <b><i>s</i></b> - поставщик  *(Supplier)*
-  <b><i>p</i></b> - товар  *(Product)*
-  <b><i>l</i></b> - локаторы полей товара *(locators)*
-  <b><i>d</i></b> - драйвер  *(Driver)*
-  <b><i>f</i></b> - поля товара  *(ProductFields)*
- <b><i>reread_locators</i></b> - перечитываль оригинальных локторов из файла

In [ ]:
p: Product = Product(s)
l: dict = s.locators['product']
d: Driver = s.driver
f: ProductFields = ProductFields(s)

### *Тестовые сценарии, функции дебаггинга*: update_locators_file()  reread_locators()

In [ ]:
"""! @debug Тестовый сценарий. Нужен для правильного распределиня категорий """

test_scenario: dict =  {
   "children": {},
   "url": "https://hbdeadsea.co.il/product-category/aromatherapy/diffusers/",
   "name": "מפיצי ריח ומילוי",
   "condition": [
     "new"
   ],
   "presta_categories": {
     "default_category": {
       "11111": "presta_category"
     },
     "addtinal categories root": [
       ""
     ]
   }
 }

### reread_locators_file

In [ ]:
def reread_locators( entity = 'product'):
    global l
    try:
        """! """
        locators_file = Path (s.supplier_abs_path, 'locators', 'product.json')
        l = s.locators [ entity ] = j_loads (locators_file) 
        return l #s.locators [ entity ]
    except Exception as ex:
        logger.error(f'ошибка ', ex)
        return False

###  Update locator file

In [ ]:
"""! @debug
    @~russian Редактор локаторов  
        Если я меняю локатор на лету - я могу заменить его в файле  
        Хорош для дебаггера и изменения локатора при изменении дизайна целевой страницы.
        Целевые страницы часто меняют дизайн, поэтому есть необходимость менять локатор "на лету".
        Перед сохранением измененного значения создается файл бекап. 
        @todo Помнить об этой особенности. Она плодит файлы!
        @param issue `str` сущность 
        @param locators `dict`: json для сохранения в файл
        @returns locator dict : перезагруженый словарь локаторов
        """
""" Перенес в класс Supplier """


def update_locators_file(locators: dict, entity: str ) -> dict:
    global s
    locators_file = Path (s.supplier_abs_path, 'locators', f'{entity}.json')
    backup_locators_file = Path (s.supplier_abs_path, 'locators', f'{gs.get_now()}_{entity}.json')
    shutil.copy2 (locators_file, backup_locators_file)
    j_dumps (locators, locators_file)
    s.locators [entity] = j_loads (locators_file)
    return s.locators [entity]


-----------------

Поля в классе f (ProductFields)

In [ ]:
#pprint(f.get_fields())

### СЦЕНАРИЙ

In [ ]:
d.get_url(test_scenario['url'])
"""! Начинаю сценарий. 1. Перешел по URL сценария """

In [ ]:
list_products_in_category: list = s.related_modules.get_list_products_in_category(s)
"""! Получил список товаров в категории """

In [ ]:
"""! Можно на них посмотреть """
#pprint(list_products_in_category)
#for url in list_products_in_category : print(url)

## Начинаю собирать поля товара
1. Перехожу по URI первого товара из списка

In [ ]:
d.get_url(list_products_in_category[0])
"""! перешел по первому url из списка """

# Эксперименты с локаторами

volume_and_price_for_100_and_sku - специальный локатор, который вытаскивает список вебэлементов

In [ ]:
"""
l['volume_and_price_for_100_and_sku'] =  {
    "attribute": None,
    "logic for attribue[AND|OR|XOR|VALUE|null]": None,
    "by": "XPATH",
    "selector": "//div[@data-widget_type='shortcode.default']",
    "logic for action[AND|OR|XOR|VALUE|null]": None,
    "use_mouse": False,
    "action": None,
    "@note": "На сайте кривой HTML, поэтому одним локатором вытескиваю 3 значения, а потом делаю парсинг. Приходит список из четырех объектов: объем, цена за 100 гр, артикул, пустой div"
 }
 """

In [ ]:
#l = update_locators_file(l, 'product')

In [ ]:
#l = reread_locators()
#pprint(l['volume_and_price_for_100_and_sku'])

In [ ]:
"""
for welement in volume_and_price_for_100_and_sku:
    pprint(welement.text)
    """

In [ ]:
#@logs_and_errors_decorator(default_return=False)
def volume_and_price_for_100_and_sku():
    """! @~russian """
    global f
    webelements: [WebElement] = d.execute_locator(l['volume_and_price_for_100_and_sku'])
        
    for webelement in webelements:
        pprint(webelement.text)
        if ('Fl.oz' and 'מ"ל' )	in webelement.text:
            """! объем """
            f.volume = webelement.text
        elif str(r'מחיר ל100 מ"ל') in webelement.text:
            """! цена за единицу товара
            @todo придумать куда
            """
            print(f'цена за единицу товара:{webelement.text}')
        elif 'מקט' in webelement.text:
            f.supplier_reference = SN.get_numbers_only(webelement.text)
        pass
    pass

In [ ]:
volume_and_price_for_100_and_sku()
"""! функция обрабатывает несколько полей: объем, цена за единицу, артикул"""

In [ ]:
pprint(f.supplier_reference)

In [ ]:
f.reference = f'{s.supplier_id}-{f.supplier_reference}'

In [ ]:
pprint(s.supplier_id)
pprint(f.reference)

## наполняю данными объект ProductFields. 
(Порядок полей в коде - алфавитный. Поля таблиц prestashop имеют свою нумерацию)

#### Заполняю поле поставщика
<a id=FieldsFormulas></a>
[отсюда можно запустить заполнение всех полей сразу](#FillAllFields)  
или:
- [Доставка](#DeliveryFields)
- [Affiliate](#AffiliateFields)
- [Description](#DescriptionFields)

дефолтные поля заполнятся при создании класс ProjectFields

### Оязательные поля

In [ ]:
f.id_supplier = s.supplier_id

In [ ]:
#reread_locators()
f.reference = d.execute_locator(l['reference'])
pprint(f.reference)

In [ ]:
pprint(l['reference'])

-----

### Доставка
<a id=DeliveryFields></a>
[наверх к списку полей](#FieldsFormulas)

In [ ]:
def field_delivery_in_stock():
    """! @~russian 
    @brief Доставка, когда товар в наличии
    @details
    """
    return d.execute_locator(l['delivery_in_stock'])


In [ ]:
def field_additional_shipping_cost():
    """! @~russian 
    @brief
    @details
    """
    
    return SN.normalize_price(d.execute_locator(l['additional_shipping_cost']))


In [ ]:
f.delivery_in_stock = field_delivery_in_stock()
f.additional_shipping_cost = field_additional_shipping_cost()

----

In [ ]:
#pprint(f.get_fields())

#print(f.delivery_in_stock)
#print(f.additional_shipping_cost)

### affiliate
<a id=AffiliateFields></a>
[наверх к списку полей](#FieldsFormulas)

In [ ]:
def field_affiliate_short_link():
    return d.execute_locator(l['affiliate_short_link'])[0]

In [ ]:
f.affiliate_short_link = field_affiliate_short_link()

### description

### описание товара
<a id=DescriptionFields></a>
[наверх к списку полей](#FieldsFormulas)

In [ ]:
def field_description():

	return str (d.execute_locator (l['description'] ) )
	pass

In [ ]:
l['description_short']: dict = {
    "attribute": None,
    "logic for attribue[AND|OR|XOR|VALUE|null]": None,
    "by": "XPATH",
    "selector": "//div[@id = 'tab-description']//p",
    "logic for action[AND|OR|XOR|VALUE|null]": None,
    "action": None,
    "use_mouse": False,
    "@note": ""
  }


In [ ]:
pprint(l['description_short'])
res = None
def field_description_short():
	res = str (d.execute_locator (l['description_short'] ) )
print(res)

In [ ]:
f.description = field_description()
f.description_short = field_description_short()

# Stop

In [ ]:
pprint(l['description'])

In [ ]:

def field_available_date():
    """! @~russian 
    @brief
    @details
    """
    return f.available_date
    pass
        
    


In [ ]:

def field_available_for_order():
    """! @~russian 
    @brief
    @details
    """
    pass



In [ ]:

def field_available_later():
    """! @~russian 
    @brief
    @details
    """
    return f.available_later
    pass


In [ ]:

def field_available_now():
    """! @~russian 
    @brief
    @details
    """
    return f.available_now
    pass




In [ ]:

def field_category_ids():
	"""! @~russian 
	@brief
	@details
	"""
	return f.category_ids
	pass
	


In [ ]:

def field_category_ids_append():
	"""! @~russian 
	@brief
	@details
	"""
	return f.category_ids_append
	pass
	
       

In [ ]:
         


def field_cache_default_attribute():
    """! @~russian 
    @brief
    @details
    """
    return f.cache_default_attribute
    pass


In [ ]:



def field_cache_has_attachments():
    """! @~russian 
    @brief
    @details
    """
    return f.cache_has_attachments
    pass	
        
   

In [ ]:
             


def field_cache_is_pack():
	"""! @~russian 
	@brief
	@details
	"""
	return f.cache_is_pack
	pass
	


In [ ]:


def field_condition(arg):
	"""! @~russian 
	@brief
	@details
	"""
	pass
	return f.condition
        


In [ ]:

def field_customizable():
	"""! @~russian 
	@brief
	@details
	"""
	return f.customizable
	pass


In [ ]:


def field_date_add():
	"""! @~russian 
	@brief
	@details
	"""
	return f.date_add
	pass
	


In [ ]:


def field_date_upd():
	"""! @~russian 
	@brief
	@details
	"""
	return f.date_upd
	pass
	


In [ ]:

"""! checked
def field_delivery_in_stock():
	return return d.execute_locator(l['delivery_in_stock'])
	pass
	"""
        


In [ ]:


def field_delivery_out_stock():

	return f.delivery_out_stock
	pass
	
                


In [ ]:


def field_depth():

	return f.depth
	pass
	


In [ ]:


def field_ean13():
	"""! @~russian 
	@brief
	@details
	"""
	return f.ean13
	pass



In [ ]:


def field_ecotax():
	"""! @~russian 
	@brief
	@details
	"""
	return f.ecotax
	pass
	
        	
                


In [ ]:


def field_height():
	"""! @~russian 
	@brief
	@details
	"""
	return f.height
	pass
	


In [ ]:


def field_how_to_use():
	"""! @~russian 
	@brief
	@details
	"""
	return f.how_to_use
	pass
	
                	


In [ ]:


def field_id_category_default():
	"""! @~russian 
	@brief
	@details
	"""
	return f.id_category_default
	pass
	


In [ ]:


def field_id_default_combination():
	"""! @~russian 
	@brief
	@details
	"""
	return f.id_default_combination
	pass
	


In [ ]:


def field_id_default_image():
	"""! @~russian 
	@brief
	@details
	"""
	return f.id_default_image
	pass
	


In [ ]:

def field_id_lang():
	"""! @~russian 
	@brief
	@details
	"""
	return f.id_lang
	pass
	


In [ ]:

def field_id_manufacturer():
	"""! @~russian 
	@brief
	@details
	"""
	return f.id_manufacturer
	pass
	


In [ ]:

def field_id_product():
	"""! @~russian 
	@brief
	@details
	"""
	return f.id_product
	pass


In [ ]:


def field_id_shop_default():
	"""! @~russian 
	@brief
	@details
	"""
	return f.id_shop_default
	pass
	


In [ ]:

def field_id_supplier():
	"""! @~russian 
	@brief
	@details
	"""
	return f.id_supplier
	pass
	


In [ ]:

def field_id_tax():
	"""! @~russian 
	@brief
	@details
	"""
	return f.id_tax
	pass
	


In [ ]:

def field_id_type_redirected():
	"""! @~russian 
	@brief
	@details
	"""
	return f.id_type_redirected
	pass



In [ ]:

def field_images_urls():
	"""! @~russian 
	@brief
	@details
	"""
	return f.images_urls
	pass
	


In [ ]:


def field_indexed():
	"""! @~russian 
	@brief
	@details
	"""
	return f.indexed
	pass
	
     

In [ ]:
   

def field_ingridients():
	"""! @~russian 
	@brief
	@details
	"""
	return f.ingridients
	pass
	


In [ ]:


def field_is_virtual():
	"""! @~russian 
	@brief
	@details
	"""
	return f.is_virtual
	pass



In [ ]:


def field_isbn():
	"""! @~russian 
	@brief
	@details
	"""
	return f.isbn
	pass
	


In [ ]:


def field_link_rewrite():
	"""! @~russian 
	@brief
	@details
	"""
	return f.link_rewrite
	pass
	
        


In [ ]:


def field_location():
	"""! @~russian 
	@brief
	@details
	"""
	return f.location
	pass
	


In [ ]:


def field_low_stock_alert():
	"""! @~russian 
	@brief
	@details
	"""
	return f.low_stock_alert
	pass
	
    


In [ ]:


def field_low_stock_threshold():
	"""! @~russian 
	@brief
	@details
	"""
	return f.low_stock_threshold
	pass
	


In [ ]:


def field_meta_description():
	"""! @~russian 
	@brief
	@details
	"""
	pass
	


In [ ]:


def field_meta_keywords():
	"""! @~russian 
	@brief
	@details
	"""
	return f.meta_keywords
	pass
	
        


In [ ]:


def field_meta_title():
	"""! @~russian 
	@brief
	@details
	"""
	return f.meta_title
	pass
	


In [ ]:



def field_minimal_quantity():
	"""! @~russian 
	@brief
	@details
	"""
	return f.minimal_quantity
	pass



In [ ]:


def field_mpn():
	"""! @~russian 
	@brief
	@details
	"""
	return f.mpn
	pass
	


In [ ]:
# 10

def field_name():
	"""! @~russian 
	@brief
	@details
	"""
	f.name =  d.execute_locator(l['name'])


In [ ]:


def field_online_only():
	"""! @~russian 
	@brief
	@details
	"""
	return f.online_only
	pass
	


In [ ]:

def field_on_sale():
	"""! @~russian 
	@brief
	@details
	"""
	return f.on_sale
	pass
	


In [ ]:


def field_out_of_stock():
	"""! @~russian 
	@brief
	@details`
	"""
	return f.out_of_stock
	pass
	



In [ ]:

def field_pack_stock_type():
	"""! @~russian 
	@brief
	@details
	"""
	return f.pack_stock_type
	pass
	


In [ ]:


def field_position_in_category():
	"""! @~russian 
	@brief
	@details
	"""
	return f.position_in_category
	pass
	
              

In [ ]:
                  


def field_price():
	"""! @~russian 
	@brief
	@details
	"""
	return f.price
	pass
	


In [ ]:


def field_product_type():
	"""! @~russian 
	@brief
	@details
	"""
	return f.product_type
	pass
	



In [ ]:

def field_quantity():
	"""! @~russian 
	@brief
	@details
	"""
	return f.quantity
	pass
	


In [ ]:


def field_quantity_discount():
	"""! @~russian 
	@brief
	@details
	"""
	return f.quantity_discount
	pass
	


In [ ]:


def field_redirect_type():
	"""! @~russian 
	@brief
	@details
	"""
	return f.redirect_type
	pass
	


In [ ]:


def field_reference():
	"""! @~russian 
	@brief
	@details
	"""
	return d.async_execute_locator(l['reference']) if async_run else d.execute_locator(l['reference'])
	pass
	


In [ ]:


def field_show_condition():
	"""! @~russian 
	@brief
	@details
	"""
	#return f.show_condition
	return 0
	pass




In [ ]:

def field_show_price():
	"""! @~russian 
	@brief
	@details
	"""
	return f.show_price
	pass



In [ ]:


def field_state():
	"""! @~russian 
	@brief
	@details
	"""
	return f.state
	pass


In [ ]:



def field_supplier_reference():
	"""! @~russian 
	@brief
	@details
	"""
	return f.supplier_reference
	pass
	



In [ ]:


def field_text_fields():
	"""! @~russian 
	@brief
	@details
	"""
	return f.text_fields
	pass
	


In [ ]:


def field_unit_price_ratio():
	"""! @~russian 
	@brief
	@details
	"""
	return f.unit_price_ratio
	pass
	



In [ ]:

def field_unity():
	"""! @~russian 
	@brief
	@details
	"""
	return f.unity
	pass
	
        


In [ ]:


def field_upc():
	"""! @~russian 
	@brief
	@details
	"""
	return f.upc
	pass
	


In [ ]:


def field_uploadable_files():
	"""! @~russian 
	@brief
	@details
	"""
	return f.uploadable_files
	pass
	


In [ ]:


def field_url_default_image():
	"""! @~russian 
	@brief
	@details
	"""
	return f.url_default_image
	pass
	


In [ ]:


def field_visibility():
	"""! @~russian 
	@brief
	@details
	"""
	return f.visibility
	pass
	


In [ ]:


def field_weight():
	"""! @~russian 
	@brief
	@details
	"""
	return f.weight
	pass
	


In [ ]:


def field_wholesale_price():
	"""! @~russian 
	@brief
	@details
	"""
	return f.wholesale_price
	pass
	


In [ ]:


def field_width():
	"""! @~russian 
	@brief
	@details
	"""
	return f.width
	pass
	
        

#### Сбор всех полей на странице
<a id=FillAllFields></a>

In [ ]:

def fill_fields(supplier: Supplier, async_run = True) -> ProductFields :
 прокручиваю страницу товара, чтобы захватить области, которые подгружаются через AJAX """

	f.active = set_field_active(f)
	f.additional_delivery_times = set_field_additional_delivery_times(f)
	f.additional_shipping_cost  = set_field_additional_shipping_cost(f)
	f.advanced_stock_management = set_field_advanced_stock_management(f)
	f.affiliate_short_link = set_field_affiliate_short_link(f)
	f.affiliate_summary = set_field_affiliate_summary(f)
	f.affiliate_summary_2 = set_field_affiliate_summary_2(f)
	f.affiliate_text = set_field_affiliate_text(f)
	f.available_date = set_field_available_date(f)
	f.available_for_order = set_field_available_for_order(f)
	f.available_later = set_field_available_later(f)
	f.available_now = set_field_available_now(f)
	f.cache_default_attribute = set_field_cache_default_attribute(f)
	f.cache_has_attachments = set_field_cache_has_attachments(f)
	f.cache_is_pack = set_field_cache_is_pack(f)
	f.category_ids_append = set_field_category_ids_append(f) #<- добавочные категории. Если надо дополнить уже внесенные
	f.category_ids = set_field_category_ids(f) #<- доп категории
	f.condition = set_field_condition(f)
	f.customizable = set_field_customizable(f)
	f.date_add = set_field_date_add(f)
	f.date_upd = set_field_date_upd(f)
	f.delivery_in_stock = set_field_delivery_in_stock(f)
	f.delivery_out_stock = set_field_delivery_out_stock(f)
	f.depth = set_field_depth(f)
	f.description = set_field_description(f)
	f.description_short = set_field_description_short(f)
	f.ean13 = set_field_ean13(f)
	f.ecotax = set_field_ecotax(f)
	f.height = set_field_height(f)
	f.how_to_use = set_field_how_to_use(f)
	f.id_category_default = set_field_id_category_default(f)
	f.id_default_combination = set_field_id_default_combination(f)
	f.id_default_image = set_field_id_default_image(f)
	f.id_lang = set_field_id_lang(f)
	f.id_manufacturer = set_field_id_manufacturer(f)
	f.id_product = set_field_id_product(f)
	f.id_shop_default = set_field_id_shop_default(f)
	f.id_supplier = set_field_id_supplier(f)
	f.id_tax = set_field_id_tax(f)
	f.id_type_redirected = set_field_id_type_redirected(f)
	f.images_urls = set_field_images_urls(f)
	f.indexed = set_field_indexed(f)
	f.ingridients = set_field_ingridients(f)
	f.is_virtual = set_field_is_virtual(f)
	f.isbn = set_field_isbn(f)
	f.link_rewrite = set_field_link_rewrite(f)
	f.location = set_field_location(f)
	f.low_stock_alert = set_field_low_stock_alert(f)
	f.low_stock_threshold = set_field_low_stock_threshold(f)
	f.meta_description = set_field_meta_description(f)
	f.meta_keywords = set_field_meta_keywords(f)
	f.meta_title = set_field_meta_title(f)
	f.minimal_quantity = set_field_minimal_quantity(f)
	f.mpn = set_field_mpn(f)
	f.name = set_field_name(f)
	f.online_only = set_field_online_only(f)
	f.on_sale = set_field_on_sale(f)
	f.out_of_stock = set_field_out_of_stock(f)
	f.pack_stock_type = set_field_pack_stock_type(f)
	f.position_in_category = set_field_position_in_category(f)
	f.price = set_field_price(f)
	f.product_type = set_field_product_type(f)
	f.quantity = set_field_quantity(f)
	f.quantity_discount = set_field_quantity_discount(f)
	f.redirect_type = set_field_redirect_type(f)
	f.reference = set_field_reference(f)
	f.show_condition = set_field_show_condition(f)
	f.show_price = set_field_show_price(f)
	f.state = set_field_state(f)
	f.supplier_reference = set_field_supplier_reference(f)
	f.text_fields = set_field_text_fields(f)
	f.unit_price_ratio = set_field_unit_price_ratio(f)
	f.unity = set_field_unity(f)
	f.upc = set_field_upc(f)
	f.uploadable_files = set_field_uploadable_files(f)
	f.url_default_image = set_field_url_default_image(f)
	f.visibility = set_field_visibility(f)
	f.weight = set_field_weight(f)
	f.wholesale_price = set_field_wholesale_price(f)
	f.width = set_field_width(f)    

	return f

In [ ]:
f.supplier_reference = d.current_url.split('item/')[-1].split('.')[0]
f.reference = f'''{s.supplier_id}-{f.supplier_reference}'''
f.name = d.execute_locator(l['name'])
pprint(f.name)
#_f.name = SF.remove_special_characters(_f.name)

In [ ]:
print(f.reference)

In [ ]:
d.execute_locator(l['additional_delivery_times'])

In [ ]:
f.price = _(l['price'])
print(f.price)

In [ ]:
product_in_presta_db = p.check_if_product_in_presta_db(f.reference)

In [ ]:
if not product_in_presta_db:
    p.add_2_prestashop(p.fields)

In [ ]:
prod_dict =             {
                "id": "",
                "id_manufacturer": "",
                "id_supplier": "",
                "id_category_default": "",
                "new": "",
                "cache_default_attribute": "",
                "id_default_image": "",
                "id_default_combination": "",
                "id_tax": "",
                "position_in_category": "",
                "type": "",
                "id_shop_default": "",
                "reference": "1005004931672329",
                "supplier_reference": "",
                "location": "",
                "width": "",
                "height": "",
                "depth": "",
                "weight": "",
                "quantity_discount": "",
                "ean13": "",
                "isbn": "",
                "upc": "",
                "mpn": "",
                "cache_is_pack": "",
                "cache_has_attachments": "",
                "is_virtual": "",
                "state": "",
                "additional_delivery_times": "",
                "delivery_in_stock": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "delivery_out_stock": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "product_type": "",
                "on_sale": "",
                "online_only": "",
                "ecotax": "",
                "minimal_quantity": "",
                "low_stock_threshold": "",
                "low_stock_alert": "",
                "price": 2439.51,
                "wholesale_price": "",
                "unity": "",
                "unit_price_ratio": "",
                "additional_shipping_cost": "",
                "customizable": "",
                "text_fields": "",
                "uploadable_files": "",
                "active": "",
                "redirect_type": "",
                "id_type_redirected": "",
                "available_for_order": "",
                "available_date": "",
                "show_condition": "",
                "condition": "",
                "show_price": "",
                "indexed": "",
                "visibility": "",
                "advanced_stock_management": "",
                "date_add": "",
                "date_upd": "",
                "pack_stock_type": "",
                "meta_description": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "meta_keywords": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "meta_title": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "link_rewrite": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "name": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "description": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "description_short": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "available_now": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "available_later": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "affiliate_short_link": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "affiliate_text": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "affiliate_summary": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "affiliate_summary_2": {
                    "language": [
                        {
                            "attrs": {
                                "id": "1"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "2"
                            },
                            "value": ""
                        },
                        {
                            "attrs": {
                                "id": "3"
                            },
                            "value": ""
                        }
                    ]
                },
                "associations": {
                    "categories": {
                        "category": {
                            "id": ""
                        }
                    },
                    "images": {
                        "image": {
                            "id": ""
                        }
                    },
                    "combinations": {
                        "combination": {
                            "id": ""
                        }
                    },
                    "product_option_values": {
                        "product_option_value": {
                            "id": ""
                        }
                    },
                    "product_features": {
                        "product_feature": {
                            "id": "",
                            "id_feature_value": ""
                        }
                    },
                    "tags": {
                        "tag": {
                            "id": ""
                        }
                    },
                    "stock_availables": {
                        "stock_available": {
                            "id": "",
                            "id_product_attribute": ""
                        }
                    },
                    "attachments": {
                        "attachment": {
                            "id": ""
                        }
                    },
                    "accessories": {
                        "product": {
                            "id": ""
                        }
                    },
                    "product_bundle": {
                        "product": {
                            "id": "",
                            "id_product_attribute": "",
                            "quantity": ""
                        }
                    }
                }
            }

In [ ]:

def fill_fields(f):
    f.id_product = ''
    f.reference = f.supplier_reference = d.current_url.split('/')[-1].split('.')[0]
    f.price = _(l['price'])[0]
    #f.price = '$151'
    f.name = _(l['name'])[0]
    f.summary = _(l['summary'])[0]
    return f

#fill_fields(f)
pass